In [ ]:
import polars as pl

In [ ]:
def load_user_embeddings() -> pl.DataFrame:
    splits = {'train': 'data/train-00000-of-00001.parquet', 'test_warm': 'data/test_warm-00000-of-00001.parquet', 'test_cold': 'data/test_cold-00000-of-00001.parquet'}
    df_train = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-User-Embeddings/" + splits["train"])
    df_test_warm = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-User-Embeddings/" + splits["test_warm"])
    df_test_cold = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-User-Embeddings/" + splits["test_cold"])
    
    df_train = df_train.with_columns(pl.lit("train").alias("split"))
    df_test_warm = df_test_warm.with_columns(pl.lit("test_warm").alias("split"))
    df_test_cold = df_test_cold.with_columns(pl.lit("test_cold").alias("split"))

    return pl.concat([df_train, df_test_warm, df_test_cold], how="vertical")

def load_dev_data() -> pl.DataFrame:
    splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
    df_train = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-Dataset/" + splits["train"])
    df_test = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-Dataset/" + splits["test"])
    df_train = df_train.with_columns(pl.lit("train").alias("split_dev"))
    df_test = df_test.with_columns(pl.lit("test").alias("split_dev"))
    return pl.concat([df_train, df_test], how="vertical")

def load_user_metadata() -> pl.DataFrame:
    df = pl.read_parquet("hf://datasets/talkpl-ai/TalkPlayData-Challenge-User-Metadata/data/all_users-00000-of-00001.parquet")
    return df

In [3]:
df = load_user_embeddings()
df.head()

user_id,cf-bpr,split
str,list[f64],str
"""5e0690dc-ffcf-41c1-94e5-2fdafc…","[0.001394, -0.001117, … 0.00007]","""train"""
"""c26f5e1c-f352-455f-aaeb-bb2d59…","[-0.000811, -0.001889, … 0.00271]","""train"""
"""3b21a6b1-338d-42c8-aa9c-236835…","[-0.003471, 0.005971, … -0.002615]","""train"""
"""a95c3a54-2962-4fa9-80ff-1bb091…","[0.000716, -0.009839, … -0.000072]","""train"""
"""585f4782-6f0f-4fce-8823-373ca6…","[-0.004334, 0.002392, … 0.001241]","""train"""


In [4]:
df.shape

(9091, 3)

# 単変量解析

# split

In [5]:
df.group_by("split").agg(pl.count())

C:\Users\ken05\AppData\Local\Temp\ipykernel_4656\1021667543.py:1: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  df.group_by("split").agg(pl.count())


split,count
str,u32
"""test_cold""",129
"""test_warm""",371
"""train""",8591


## user_id

In [6]:
df["user_id"][0]

'5e0690dc-ffcf-41c1-94e5-2fdafc4bd4ed'

In [7]:
df["user_id"].value_counts().sort("count", descending=True).head()

user_id,count
str,u32
"""fa891983-f810-4769-8d11-3d7315…",2
"""80fd5bb2-e9d9-4a02-b86d-e7b2aa…",2
"""39fdc5d3-37c0-4f7a-9bd5-0a7a93…",2
"""b3475bae-7bc5-49b2-a2d4-326d8a…",2
"""b3a298af-dd88-4809-8682-e004fe…",2


In [8]:
user_id_count = df["user_id"].value_counts().rename({"count": "appearance"})
user_id_count.group_by("appearance").agg(pl.len())

appearance,len
u32,u32
1,8349
2,371


In [9]:
user_id_count_is2 = user_id_count.filter(pl.col("appearance") == 2)
df.filter((pl.col("split") == "test_warm") & (pl.col("user_id").is_in(user_id_count_is2["user_id"]))).shape

C:\Users\ken05\AppData\Local\Temp\ipykernel_4656\1698753904.py:2: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df.filter((pl.col("split") == "test_warm") & (pl.col("user_id").is_in(user_id_count_is2["user_id"]))).shape


(371, 3)

In [10]:
df.group_by(["split", "user_id"]).agg(pl.len()).sort("len", descending=True).head()

split,user_id,len
str,str,u32
"""test_warm""","""704cf4a0-bad3-4e4b-b165-7b77b6…",1
"""train""","""21f98a49-610d-4413-bd48-cb5efe…",1
"""train""","""89d056fe-2af1-4842-bf99-40a164…",1
"""train""","""50e2e609-cbc4-4144-8097-d95ec5…",1
"""test_warm""","""a317f5c9-616b-4482-b62a-01fda1…",1


# cf-bpr

In [11]:
df["cf-bpr"].head()

cf-bpr
list[f64]
"[0.001394, -0.001117, … 0.00007]"
"[-0.000811, -0.001889, … 0.00271]"
"[-0.003471, 0.005971, … -0.002615]"
"[0.000716, -0.009839, … -0.000072]"
"[-0.004334, 0.002392, … 0.001241]"
"[0.000863, -0.002547, … 0.00004]"
"[-0.000148, -0.000918, … -0.007247]"
"[0.012863, 0.000981, … -0.008897]"
"[0.003417, 0.00174, … 0.004345]"


In [12]:
df.with_columns(pl.col("cf-bpr").list.len().alias("emb_len")).select(pl.col("emb_len").unique())

emb_len
u32
0
128


In [13]:
cf_bpr_len = df.with_columns(pl.col("cf-bpr").list.len().alias("emb_len"))
cf_bpr_len.select(["emb_len", "split"]).group_by(["emb_len", "split"]).agg(pl.len())

emb_len,split,len
u32,str,u32
128,"""test_cold""",25
128,"""train""",8591
0,"""test_cold""",104
128,"""test_warm""",371


In [14]:
cf_bpr_len.filter((pl.col("emb_len") == 128) & (pl.col("split") == "test_cold")).head(10)

user_id,cf-bpr,split,emb_len
str,list[f64],str,u32
"""0b26e17b-0627-45e3-baa1-aa285a…","[0.001365, 0.002224, … -0.002632]","""test_cold""",128
"""9a003c54-6d2d-416e-bef1-dc1d05…","[-0.000827, -0.002198, … -0.001971]","""test_cold""",128
"""cc31cceb-9477-45af-97f6-c64889…","[-0.0035, -0.004494, … -0.00155]","""test_cold""",128
"""1b6cf64c-611b-419e-b9d6-f43f7d…","[0.00001, 0.002219, … -0.00284]","""test_cold""",128
"""fba103ec-2939-402e-8560-5a57a8…","[-0.00345, -0.002423, … 0.001625]","""test_cold""",128
"""b74bae1a-56ad-4cc5-95c1-f72bd8…","[0.000072, 0.001815, … -0.00379]","""test_cold""",128
"""777529e6-7f1a-4b7c-b30b-a2bba9…","[-0.014244, 0.020431, … 0.005433]","""test_cold""",128
"""0cdbcde9-afb1-4ec7-8356-51d324…","[-0.011164, 0.008934, … -0.001034]","""test_cold""",128
"""db93bf83-27ca-47e8-81b3-b43396…","[-0.003385, 0.003455, … 0.00185]","""test_cold""",128


In [15]:
cf_bpr_len.filter((pl.col("emb_len") == 0) & (pl.col("split") == "test_cold")).head(10)

user_id,cf-bpr,split,emb_len
str,list[f64],str,u32
"""e1dac513-ee13-40a1-9bb1-fa69e0…",[],"""test_cold""",0
"""1b8d08b3-f103-4af7-a87b-f3fee8…",[],"""test_cold""",0
"""bb95b750-f90c-4220-bf14-544345…",[],"""test_cold""",0
"""b4c46ecb-3085-4525-875b-a8fa53…",[],"""test_cold""",0
"""653f918b-28b5-46cc-8cbd-bea194…",[],"""test_cold""",0
"""7fb1c25a-8b8a-4270-a1b9-ca5e54…",[],"""test_cold""",0
"""9c9c74f3-e20b-4976-969b-cdcf43…",[],"""test_cold""",0
"""4a79be1f-321a-4352-8ddf-f3c08a…",[],"""test_cold""",0
"""2849bce6-9b66-402f-b49b-948e76…",[],"""test_cold""",0


### cf-bprの0次元の謎をほかのデータと照らし合わせてみる

In [16]:
df_metadata = load_user_metadata()
df_dev = load_dev_data()

df_all = (
    df
    .join(df_metadata, on="user_id", how="left")
    .join(df_dev, on="user_id", how="left")
)
df_all.head(3)

user_id,cf-bpr,split,age,age_group,country_code,country_name,gender,session_id,session_date,user_profile,conversation_goal,conversations,goal_progress_assessments,split_dev
str,list[f64],str,i64,str,str,str,str,str,str,struct[9],struct[3],list[struct[4]],list[struct[2]],str
"""5e0690dc-ffcf-41c1-94e5-2fdafc…","[0.001394, -0.001117, … 0.00007]","""train""",19,"""10s""","""MX""","""Mexico""","""female""","""e46b0b82-e42c-4a90-ac2f-257e87…","""2014-10-01""","{19,""10s"",""MX"",""Mexico"",""female"",""English"",""Western Anglophone Alternative and Classic Rock"",""5e0690dc-ffcf-41c1-94e5-2fdafc4bd4ed"",""train_warm""}","{""H"",""find multiple songs from 2010s indie electronic/pop artists with specific sonic characteristics or vocal styles."",""HL""}","[{""Can you show me indie pop artists from the 2010s that feature strong female vocals, similar to MisterWives or Lykke Li?"",""user"","""",1}, {""c22144f2-a51a-46b6-a489-6b4c7e3a4e08"",""music"",""Glad you liked The Doors! Sticking with that Classic Rock/Alternative Rock energy you enjoy, I'm recommending ""Beating Heart Baby"" by Head Automatica. It's got that lively, rock feel that I think you'll appreciate."",1}, … {""Awesome! If you're ready for something with a similar energy but a bit more synth-pop, you should check out ""DNA"" by Empire of the Sun. It's got a really cool, upbeat sound!"",""assistant"","""",8}]","[{null,1}, {""DOES_NOT_MOVE_TOWARD_GOAL"",2}, … {""MOVES_TOWARD_GOAL"",8}]","""train"""
"""c26f5e1c-f352-455f-aaeb-bb2d59…","[-0.000811, -0.001889, … 0.00271]","""train""",25,"""20s""","""PL""","""Poland""","""female""","""f0408484-2de8-4fd8-ae67-f9f5dc…","""2013-07-13""","{25,""20s"",""PL"",""Poland"",""female"",""English"",""American Classic Rock"",""c26f5e1c-f352-455f-aaeb-bb2d59821740"",""train_warm""}","{""H"",""find multiple songs from classic rock artists known for their psychedelic and blues influences, or from modern indie folk artists recognized for their distinctive vocal harmonies and storytelling."",""HL""}","[{""I'm interested in exploring the unique psychedelic blues sound of The Doors. Can you show me some key tracks from their albums released between 1967 and 1971?"",""user"","""",1}, {""5c1231ef-616f-45a8-9e4b-ab027f1b60e3"",""music"",""The listener enjoyed the previous Doors track, confirming their preference for the band and classic rock. ""Love Me Two Times"" is another quintessential Doors song that fits the classic rock genre perfectly and should appeal to their favorite artist."",1}, … {""So glad you liked ""Respect""! Keeping with that classic American rock sound, but maybe a little more mellow, how about ""Dance With Me"" by Orleans? It's a great soft rock classic from the 70s."",""assistant"","""",8}]","[{null,1}, {""MOVES_TOWARD_GOAL"",2}, … {""DOES_NOT_MOVE_TOWARD_GOAL"",8}]","""train"""
"""3b21a6b1-338d-42c8-aa9c-236835…","[-0.003471, 0.005971, … -0.002615]","""train""",23,"""20s""","""PL""","""Poland""","""male""","""b60d2ccd-5882-468c-ab9a-082f20…","""2018-01-08""","{23,""20s"",""PL"",""Poland"",""male"",""English"",""American Pop Culture (80s-inspired)"",""3b21a6b1-338d-42c8-aa9c-236835995844"",""train_warm""}","{""H"",""Identify a specific artist or find a specific song from them, based on vague memories of a distinctive electronic, synth-heavy soundtrack style."",""LH""}","[{""I'm trying to find some music that has a very retro synth sound, almost like it's from an old sci-fi movie."",""user"","""",1}, {""18811062-0234-4d06-921b-6bbc132e64b4"",""music"",""Glad to hear you enjoyed the last track! Since you're loving the synthwave from Kyle Dixon & Michael Stein, ""Hanging Lights"" is another perfect fit from the Stranger Things soundtrack, keeping that immersive electronic, 80s vibe going for you."",1}, … {""Fantastic! Glad we're hitting all the right notes for you. To keep that awesome synthwave journey going, I've got ""Run Away"" next, also by Kyle Dixon & Michael Stein. It's got that perfect 80s soundtrack vibe."",""assistant"","

In [ ]:
df_metadata["user_id"].value_counts().sort("count", descending=True).head()

AttributeError: 'DataFrame' object has no attribute 'user_id'